# Hospital Meal Planning Optimization

**Objective:** Minimize food cost while meeting clinical nutrition targets and maximizing patient satisfaction

**Model overview:**
- 41 ingredients across 5 food categories (Fruit, Vegetable, Grain, Protein, Fat)
- 5 adult clinical profiles (Normal Male, Normal Female, Diabetic, High Cholesterol, High BP (DASH))
- 3-day planning with 3 meals/day = 9 meal slots
- Per-day micronutrient and macronutrient compliance (protein, fat, carbs, fiber, minerals, vitamins)
- Per-meal calorie windows based on Women's College Hospital estimates (35% Breakfast / 40% Lunch / 25% Dinner)

**Three solvers + LP relaxation:**
1. **Solver 0 (Cost-only):** Minimizes food cost only (same-day and cross-day repeats are applicable)
2. **Solver 1 (Hard repeat constraints):** Minimize food cost with hard no-repeat rules (no same-day repeats, cross-day repeats with max 2 days per ingredient)
3. **LP Relaxation:** Continuous relaxation of Solver 2 for sensitivity analysis (shadow prices on binding constraints)


## Step 1 — Install & Import Libraries

- openpyxl - Read, write and modify Excel files https://pypi.org/project/openpyxl/
- scipy - Algorithms for scientific computing in Python for optimization https://docs.scipy.org/doc/scipy/tutorial/optimize.html#mixed-integer-linear-programming
- numpy - Numerical operations for scientific computing in Python https://numpy.org/doc/stable/user/basics.creation.html
- plotly - Graphing library for Python https://plotly.com/python/

In [ ]:
!pip install openpyxl scipy numpy plotly -q

In [ ]:
import numpy as np
import openpyxl
import time
import warnings
from scipy.optimize import milp, linprog, LinearConstraint, Bounds
from scipy.sparse import csr_matrix
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings('ignore')
print('Libraries loaded.')

Libraries loaded.


## Step 2 — Load Data from Excel

All model data lives in a single Excel workbook with 10 sheets. Each cell below reads one sheet.

In [ ]:
# Data files live alongside this notebook.
# (If running in Colab, upload Meal_Planner_Data.xlsx to the session first.)


In [ ]:
filename = "Meal_Planner_Data.xlsx"
workbook = openpyxl.load_workbook(filename)


### 2a — Ingredients (41 items)

Each ingredient has a food category, price per gram, eligible meal slots (B/L/D), meal-type-specific portion bounds (min/max grams).

In [ ]:
sheet = workbook['Ingredients']
rows = [row for row in sheet.iter_rows(min_row=2, values_only=True) if row[0] is not None]

ingredient_names = [row[1] for row in rows]
categories       = [row[2] for row in rows]
price_per_gram   = np.array([float(row[3]) for row in rows])
eligible_slots   = [set(row[4].split(',')) for row in rows]
num_ingredients  = len(ingredient_names)

# Portion bounds are meal-type-specific: (B_min, B_max, L_min, L_max, D_min, D_max)
portion_bounds = []
for row in rows:
    portion_bounds.append((float(row[5]), float(row[6]), float(row[7]), float(row[8]), float(row[9]), float(row[10])))

def get_portion_minmax(ingredient_idx, meal_idx):
    """Return (min_grams, max_grams) for a given ingredient at a given meal slot."""
    meal_type = meal_type_codes[meal_idx]
    bounds = portion_bounds[ingredient_idx]
    if meal_type == 'B': return bounds[0], bounds[1]
    elif meal_type == 'L': return bounds[2], bounds[3]
    else: return bounds[4], bounds[5]

print(f'Loaded {num_ingredients} ingredients with categories:')
for category in ['Fruit', 'Vegetable', 'Grain', 'Protein', 'Fat']:
    count = sum(1 for c in categories if c == category)
    print(f'  {category}: {count}')

Loaded 41 ingredients with categories:
  Fruit: 10
  Vegetable: 6
  Grain: 9
  Protein: 10
  Fat: 6


### 2b — Nutrient Density Matrix

- Each row is an ingredient
- Each column is a macro or micro nutrient in g or mg
- Some nutrients (VitA, Sugar) are "monitor only". They are tracked but not constrained.

In [ ]:
#open Nutrients sheet from Excel
sheet = workbook['Nutrients']
nutrient_rows = [row for row in sheet.iter_rows(min_row=2, values_only=True) if row[0] is not None]


nutrient_matrix = np.array([[float(nutrient) for nutrient in row[1:14]] for row in nutrient_rows])

nutrient_names = ['Cal', 'Prot', 'Fat', 'Carb', 'Fib', 'Ca', 'Fe', 'Na', 'K', 'VitA', 'VitD', 'Sugar', 'SatFat']
nutrient_units = ['kcal', 'g', 'g', 'g', 'g', 'mg', 'mg', 'mg', 'mg', 'mcg', 'mcg', 'g', 'g']

# Monitor flags for nutrients with flag=1 are tracked but not hard constraints
sheet = workbook['Monitor_Flags']
monitored_nutrients = set()
for row in sheet.iter_rows(min_row=2, values_only=True):
    if row and row[0] is not None and row[1] == 1:
        monitored_nutrients.add(int(row[0]))

print(f'Nutrient matrix shape: {nutrient_matrix.shape} (ingredients × nutrients)')
print(f'Monitored only: {[nutrient_names[monitored_nutrient] for monitored_nutrient in monitored_nutrients]}')

Nutrient matrix shape: (41, 13) (ingredients × nutrients)
Monitored only: ['VitA', 'Sugar']


### 2c — Clinical Profiles

- **Daily nutrient bounds** (per day): protein, fat, carbs, fiber, minerals, vitamins.

- **Per-meal calorie bounds**: Based on Women's College Hospital estimates with a 35/40/25 split across Breakfast/Lunch/Dinner. Snack calories were redistributed into the three meals to keep it to the three meals a day.

- **Diabetic per-meal carb constraint**: 45–60g carbs per meal to prevent blood sugar spikes.

In [ ]:
# Daily nutrient bounds
sheet = workbook['Profiles_Daily']
profiles = {}

# Nutrients we track daily
daily_nutrient_indices = [1, 2, 3, 4, 5, 6, 7, 8, 10, 12]  # indices into the 13-column nutrient matrix
daily_nutrient_names = ['Prot', 'Fat', 'Carb', 'Fib', 'Ca', 'Fe', 'Na', 'K', 'VitD', 'SatFat']

for row in sheet.iter_rows(min_row=2, values_only=True):
    if row[0] is None: continue
    name = row[0]
    vals = [float(v) if v is not None else 0 for v in row[1:23]]

    #Nutrient map
    profiles[name] = {
        'min': [0, vals[2], vals[4], vals[6], vals[8], vals[10], vals[12], vals[14], vals[16], 0, vals[18], 0, vals[20]],
        'max': [1e9, vals[3], vals[5], vals[7], vals[9], vals[11], vals[13], vals[15], vals[17], 1e9, vals[19], 1e9, vals[21]]
    }

# Per meal calorie bounds
sheet = workbook['Profiles_Meal']
meal_cal_bounds = {}
for row in sheet.iter_rows(min_row=2, values_only=True):
    if row[0] is None: continue
    meal_cal_bounds[row[0]] = {
        'B': (float(row[1]), float(row[2])),
        'L': (float(row[3]), float(row[4])),
        'D': (float(row[5]), float(row[6]))
    }

# Diabetic per-meal carb constraint
sheet = workbook['Diabetic_Meal_Carb']
diabetic_meal_carb = {}
for row in sheet.iter_rows(min_row=2, values_only=True):
    if row[0] is None: continue
    diabetic_meal_carb[row[0]] = (float(row[1]), float(row[2]))

profile_names = list(profiles.keys())
print(f'Loaded {len(profiles)} profiles:\n')
for profile in profiles:
    bd = profiles[profile]
    mc = meal_cal_bounds[profile]
    print(f'  {profile}:')
    print(f'    Daily: Prot={bd["min"][1]:.0f}-{bd["max"][1]:.0f}g  Fat={bd["min"][2]:.0f}-{bd["max"][2]:.0f}g  Carb={bd["min"][3]:.0f}-{bd["max"][3]:.0f}g')
    print(f'    Per-meal cal: B={mc["B"]}  L={mc["L"]}  D={mc["D"]}')
    if profile in diabetic_meal_carb:
        print(f'    Per-meal carb: {diabetic_meal_carb[profile][0]:.0f}-{diabetic_meal_carb[profile][1]:.0f}g')

Loaded 5 profiles:

  Normal Male:
    Daily: Prot=75-112g  Fat=66-100g  Carb=275-412g
    Per-meal cal: B=(700.0, 875.0)  L=(800.0, 1000.0)  D=(500.0, 625.0)
  Normal Female:
    Daily: Prot=60-90g  Fat=54-80g  Carb=220-330g
    Per-meal cal: B=(560.0, 700.0)  L=(640.0, 800.0)  D=(400.0, 500.0)
  Diabetic:
    Daily: Prot=90-132g  Fat=60-90g  Carb=130-240g
    Per-meal cal: B=(560.0, 700.0)  L=(640.0, 800.0)  D=(400.0, 500.0)
    Per-meal carb: 45-60g
  High Cholesterol:
    Daily: Prot=60-90g  Fat=0-54g  Carb=220-330g
    Per-meal cal: B=(560.0, 700.0)  L=(640.0, 800.0)  D=(400.0, 500.0)
  DASH:
    Daily: Prot=72-108g  Fat=48-72g  Carb=200-300g
    Per-meal cal: B=(560.0, 700.0)  L=(640.0, 800.0)  D=(400.0, 500.0)


## Step 3 — Meal Structure & Category Indices

The 3-day plan has 9 meal slots (Breakfast, Lunch, Dinner) - Day 1 (B, L, D), Day 2 (B, L, D), Day 3 (B, L, D).
We also group ingredients by food category for use in constraints.

In [ ]:
NUM_MEALS = 9
NUM_DAYS  = 3

meal_type_codes  = ['B', 'L', 'D'] * 3
day_meal_indices = [[0, 1, 2], [3, 4, 5], [6, 7, 8]]
breakfast_slots  = [0, 3, 6]
lunch_slots      = [1, 4, 7]
dinner_slots     = [2, 5, 8]
meal_labels      = ['Day1-B', 'Day1-L', 'Day1-D', 'Day2-B', 'Day2-L', 'Day2-D', 'Day3-B', 'Day3-L', 'Day3-D']

# Category indices
fruit_indices     = [i for i, c in enumerate(categories) if c == 'Fruit']
vegetable_indices = [i for i, c in enumerate(categories) if c == 'Vegetable']
grain_indices     = [i for i, c in enumerate(categories) if c == 'Grain']
protein_indices   = [i for i, c in enumerate(categories) if c == 'Protein']
fat_indices       = [i for i, c in enumerate(categories) if c == 'Fat']

# Breakfast grains - Grain items eligible for Breakfast
breakfast_grain_indices = [i for i in grain_indices if 'B' in eligible_slots[i]]

# Lunch/Dinner grains - Grain items eligible for Lunch or Dinner
lunch_dinner_grain_indices = [i for i in grain_indices if 'L' in eligible_slots[i] or 'D' in eligible_slots[i]]

print(f'Categories: Fruit={len(fruit_indices)}, Vegetable={len(vegetable_indices)}, '
      f'Grain={len(grain_indices)}, Protein={len(protein_indices)}, Fat={len(fat_indices)}')
print(f'Breakfast grains: {[ingredient_names[i] for i in breakfast_grain_indices]}')
print(f'L/D grains: {[ingredient_names[i] for i in lunch_dinner_grain_indices]}')

Categories: Fruit=10, Vegetable=6, Grain=9, Protein=10, Fat=6
Breakfast grains: ['Oatmeal', 'Whole Wheat Bread', 'Granola']
L/D grains: ['Sweet Potato', 'White Rice', 'Brown Rice', 'Whole Wheat Bread', 'Quinoa', 'Spaghetti', 'Barley']


## Step 4 — Culinary Incompatibility Rules

In other to prevent the model from sharing incompatible food combinations, we had to set some culinary rules for the optimizer. Certain ingredient pairings are blocked because they don't make sense together on a plate.

In [ ]:
sheet = workbook['Culinary_Rules']
incompatible_all_meals = []
incompatible_breakfast = []
for row in sheet.iter_rows(min_row=2, values_only=True):
    if row[0] is None: continue
    idx_a = ingredient_names.index(row[0])
    idx_b = ingredient_names.index(row[1])
    if row[2] == 'All meals':
        incompatible_all_meals.append((idx_a, idx_b))
    else:
        incompatible_breakfast.append((idx_a, idx_b))

print(f'Incompatible pairs (all meals): {len(incompatible_all_meals)}')
for a, b in incompatible_all_meals:
    print(f'  {ingredient_names[a]} + {ingredient_names[b]}')
print(f'\nIncompatible pairs (breakfast only): {len(incompatible_breakfast)}')
for a, b in incompatible_breakfast:
    print(f'  {ingredient_names[a]} + {ingredient_names[b]}')

Incompatible pairs (all meals): 13
  Oatmeal + Beef
  Oatmeal + Chicken
  Oatmeal + Salmon
  Oatmeal + Turkey
  Oatmeal + Tofu
  Granola + Beef
  Granola + Chicken
  Granola + Salmon
  Granola + Turkey
  Granola + Tofu
  Flaxseeds + Chia Seeds
  Tofu + Whole Wheat Bread
  Lentils + Whole Wheat Bread

Incompatible pairs (breakfast only): 4
  Peanut Butter + Salmon
  Peanut Butter + Turkey
  Greek Yogurt + Whole Wheat Bread
  Egg + Granola


## Step 6 — Decision Variables


| Variable | Boundaries | Definition |
|---|---|---|
| `x[i,j]` | {0, 1} | Is ingredient *i* selected at meal *j*? (eligible pairs only) |
| `q[i,j]` | ≥ 0 (continuous) | Grams of ingredient *i* at meal *j* (eligible pairs only) |
| `a[i,d]` | {0, 1} | Does ingredient *i* appear on day *d*? |
| `r[i]` | {0, 1} | Does ingredient *i* appear on multiple days? |


In [ ]:
# Build eligible ingredient and meal pairs
eligible_pairs = []
pair_index = {}

for i in range(num_ingredients):
    for j in range(NUM_MEALS):
        if meal_type_codes[j] in eligible_slots[i]:
            pair_index[(i, j)] = len(eligible_pairs)
            eligible_pairs.append((i, j))

num_eligible_pairs = len(eligible_pairs)

# Variable layout:
#   [0, P)         : x[i,j] selection binary
#   [P, 2P)        : q[i,j] quantity continuous
#   [2P, 2P+N*D)   : a[i,d] is ingredient present on this day binary
#   [2P+N*D, end)  : r[i] cross day repeat binary
total_vars = 2 * num_eligible_pairs + num_ingredients * NUM_DAYS + num_ingredients

def idx_selection(i, j):      return pair_index[(i, j)]
def idx_quantity(i, j):        return num_eligible_pairs + pair_index[(i, j)]
def idx_day_presence(i, d):    return 2 * num_eligible_pairs + i * NUM_DAYS + d
def idx_cross_day_repeat(i):   return 2 * num_eligible_pairs + num_ingredients * NUM_DAYS + i

PLATE_RATIO = 1.5  # Density-adjusted plate ratio

print(f'Eligible pairs: {num_eligible_pairs} / {num_ingredients * NUM_MEALS}')
print(f'Total variables: {total_vars:,}')

Eligible pairs: 261 / 369
Total variables: 686


## Step 7 — Constraints

This function builds all constraints used by all solvers.

**Constraints:**
- **C1**:  Daily Nutrient Bounds (protein, fat, carbs, fiber, minerals)
- **C2**: Per-Meal Calories (profile-specific, replaces daily calorie constraint)
- **C3**: Diabetic Per-Meal Carb Limit (45–60g per meal, Diabetic profile only)
- **C4**: Meal Complexity (1–4 non-fruit items per meal)
- **C5**: Culinary Incompatibilities
- **C6**: Breakfast Structure (≥1 protein, exactly 1 grain, exactly 1 fat)
- **C7**: Lunch/Dinner Structure (≥1 protein, ≥1 vegetable, 1–2 grains, plate ratio)
- **C8**: Portion Bounds
- **C9**: Day Presence Linking


In [ ]:
def build_constraints(profile_name):
    """Build the constraint matrix for a given clinical profile.
    Returns (rows_A, lower_bounds, upper_bounds, labels)."""
    bounds_dict = profiles[profile_name]
    nutrient_min = np.array(bounds_dict['min'])
    nutrient_max = np.array(bounds_dict['max'])

    rows_A = []
    lower_bounds = []
    upper_bounds = []
    labels = []

    def add_constraint(row, lo, hi, label=''):
        rows_A.append(row)
        lower_bounds.append(lo)
        upper_bounds.append(hi)
        labels.append(label)

    # C1 - Per-day nutrient bounds
    for day in range(NUM_DAYS):
        for k in range(13):
            if k in monitored_nutrients or k == 0:
                continue
            row = np.zeros(total_vars)
            for meal in day_meal_indices[day]:
                for ing in range(num_ingredients):
                    if (ing, meal) in pair_index:
                        row[idx_quantity(ing, meal)] += nutrient_matrix[ing, k]
            add_constraint(row, nutrient_min[k], nutrient_max[k],
                        f'Day {day+1} {nutrient_names[k]} [{nutrient_min[k]:.0f}-{nutrient_max[k]:.0f}]')

    # C2 - Per-meal calorie bounds
    if profile_name in meal_cal_bounds:
        mcb = meal_cal_bounds[profile_name]
        for meal in range(NUM_MEALS):
            cal_lo, cal_hi = mcb[meal_type_codes[meal]]
            row = np.zeros(total_vars)
            for ing in range(num_ingredients):
                if (ing, meal) in pair_index:
                    row[idx_quantity(ing, meal)] += nutrient_matrix[ing, 0]
            add_constraint(row, cal_lo, cal_hi,
                        f'{meal_labels[meal]} Cal [{cal_lo:.0f}-{cal_hi:.0f}]')

    # C3 - Diabetic per-meal carb constraint
    if profile_name in diabetic_meal_carb:
        carb_lo, carb_hi = diabetic_meal_carb[profile_name]
        for meal in range(NUM_MEALS):
            row = np.zeros(total_vars)
            for ing in range(num_ingredients):
                if (ing, meal) in pair_index:
                    row[idx_quantity(ing, meal)] += nutrient_matrix[ing, 3]
            add_constraint(row, carb_lo, carb_hi,
                        f'{meal_labels[meal]} Carb [{carb_lo:.0f}-{carb_hi:.0f}]')

    #  C4 - Exactly 1 fruit per meal
    for meal in range(NUM_MEALS):
        row = np.zeros(total_vars)
        for ing in fruit_indices:
            if (ing, meal) in pair_index:
                row[idx_selection(ing, meal)] = 1
        add_constraint(row, 1, 1, f'{meal_labels[meal]} exactly 1 fruit')

        # C4 - Meal complexity (1-4 non-fruit items)
    for meal in range(NUM_MEALS):
        row = np.zeros(total_vars)
        for ing in range(num_ingredients):
            if categories[ing] != 'Fruit' and (ing, meal) in pair_index:
                row[idx_selection(ing, meal)] = 1
        add_constraint(row, 1, 4, f'{meal_labels[meal]} 1-4 non-fruit items')


    # C5 - Culinary incompatibilities (all meals)
    for (ing_a, ing_b) in incompatible_all_meals:
        for meal in range(NUM_MEALS):
            if (ing_a, meal) in pair_index and (ing_b, meal) in pair_index:
                row = np.zeros(total_vars)
                row[idx_selection(ing_a, meal)] = 1
                row[idx_selection(ing_b, meal)] = 1
                add_constraint(row, 0, 1,
                            f'{meal_labels[meal]} no {ingredient_names[ing_a]}+{ingredient_names[ing_b]}')

    # Culinary incompatibilities (breakfast only)
    for (ing_a, ing_b) in incompatible_breakfast:
        for meal in breakfast_slots:
            if (ing_a, meal) in pair_index and (ing_b, meal) in pair_index:
                row = np.zeros(total_vars)
                row[idx_selection(ing_a, meal)] = 1
                row[idx_selection(ing_b, meal)] = 1
                add_constraint(row, 0, 1,
                            f'{meal_labels[meal]} no {ingredient_names[ing_a]}+{ingredient_names[ing_b]}')


    # C6 - Breakfast structure
    for meal in breakfast_slots:
        row = np.zeros(total_vars)
        for ing in protein_indices:
            if (ing, meal) in pair_index:
                row[idx_selection(ing, meal)] = 1
        add_constraint(row, 1, 4, f'{meal_labels[meal]} >=1 protein')

        row = np.zeros(total_vars)
        for ing in breakfast_grain_indices:
            if (ing, meal) in pair_index:
                row[idx_selection(ing, meal)] = 1
        add_constraint(row, 1, 1, f'{meal_labels[meal]} =1 grain')

        row = np.zeros(total_vars)
        for ing in fat_indices:
            if (ing, meal) in pair_index:
                row[idx_selection(ing, meal)] = 1
        add_constraint(row, 1, 1, f'{meal_labels[meal]} =1 fat')

    # C7 - Lunch/Dinner structure
    for meal in lunch_slots + dinner_slots:
        row = np.zeros(total_vars)
        for ing in protein_indices:
            if (ing, meal) in pair_index:
                row[idx_selection(ing, meal)] = 1
        add_constraint(row, 1, 4, f'{meal_labels[meal]} >=1 protein')

        row = np.zeros(total_vars)
        for ing in vegetable_indices:
            if (ing, meal) in pair_index:
                row[idx_selection(ing, meal)] = 1
        add_constraint(row, 1, 4, f'{meal_labels[meal]} >=1 vegetable')

        row = np.zeros(total_vars)
        for ing in lunch_dinner_grain_indices:
            if (ing, meal) in pair_index:
                row[idx_selection(ing, meal)] = 1
        add_constraint(row, 1, 2, f'{meal_labels[meal]} 1-2 grains')

        row = np.zeros(total_vars)
        for ing in vegetable_indices:
            if (ing, meal) in pair_index:
                row[idx_quantity(ing, meal)] = PLATE_RATIO
        for ing in protein_indices:
            if (ing, meal) in pair_index:
                row[idx_quantity(ing, meal)] = -1.0
        for ing in lunch_dinner_grain_indices:
            if (ing, meal) in pair_index:
                row[idx_quantity(ing, meal)] = -1.0
        add_constraint(row, 0, np.inf, f'{meal_labels[meal]} plate ratio')

    # C8 - Portion bounds
    for (ing, meal) in eligible_pairs:
        portion_min, portion_max = get_portion_minmax(ing, meal)
        row_upper = np.zeros(total_vars)
        row_upper[idx_quantity(ing, meal)] = 1
        row_upper[idx_selection(ing, meal)] = -portion_max
        add_constraint(row_upper, -np.inf, 0,
                    f'{meal_labels[meal]} {ingredient_names[ing]} max {portion_max:.0f}g')
        row_lower = np.zeros(total_vars)
        row_lower[idx_quantity(ing, meal)] = 1
        row_lower[idx_selection(ing, meal)] = -portion_min
        add_constraint(row_lower, 0, np.inf,
                    f'{meal_labels[meal]} {ingredient_names[ing]} min {portion_min:.0f}g')

    # C9 - Day presence linking
    for ing in range(num_ingredients):
        for day in range(NUM_DAYS):
            row = np.zeros(total_vars)
            for meal in day_meal_indices[day]:
                if (ing, meal) in pair_index:
                    row[idx_selection(ing, meal)] = 1
            row[idx_day_presence(ing, day)] = -3
            add_constraint(row, -np.inf, 0, f'Day {day+1} {ingredient_names[ing]} presence upper')
            row2 = np.zeros(total_vars)
            row2[idx_day_presence(ing, day)] = 1
            for meal in day_meal_indices[day]:
                if (ing, meal) in pair_index:
                    row2[idx_selection(ing, meal)] = -1
            add_constraint(row2, -np.inf, 0, f'Day {day+1} {ingredient_names[ing]} presence lower')

    # Cross-day repeat detection
    for ing in range(num_ingredients):
        row = np.zeros(total_vars)
        for day in range(NUM_DAYS):
            row[idx_day_presence(ing, day)] = 1
        row[idx_cross_day_repeat(ing)] = -1
        add_constraint(row, -np.inf, 1, f'{ingredient_names[ing]} cross-day repeat')

    return rows_A, lower_bounds, upper_bounds, labels

print('Constraints defined.')

Constraints defined.


## Step 8 — Variable Bounds & Integrality

Sets up the variable domains, binary for selection/presence/repeat, continuous for quantity.

In [ ]:
def build_variable_bounds():
    """Return lower bounds, upper bounds, and integrality vector."""
    lower = np.zeros(total_vars)
    upper = np.full(total_vars, np.inf)

    # x[i,j] binary
    for idx in range(num_eligible_pairs):
        upper[idx] = 1.0

    # a[i,d] binary
    for ing in range(num_ingredients):
        for day in range(NUM_DAYS):
            upper[idx_day_presence(ing, day)] = 1.0
        upper[idx_cross_day_repeat(ing)] = 1.0

    integrality = np.zeros(total_vars)
    integrality[0:num_eligible_pairs] = 1                                             # x binary
    integrality[2*num_eligible_pairs : 2*num_eligible_pairs + num_ingredients*NUM_DAYS] = 1  # a binary
    integrality[2*num_eligible_pairs + num_ingredients*NUM_DAYS :] = 1                 # row binary

    return lower, upper, integrality

print('Variable bounds defined.')

Variable bounds defined.


## Step 9 — Solver 0: Cost Only

**Objective:** Minimize food cost only.


In [ ]:
def solve_cost_only(profile_name, time_limit=200):
    """Solver 0: Pure cost minimization — no repeat tracking, no penalties."""

    cost = np.zeros(total_vars)
    for (ing, meal) in eligible_pairs:
        cost[idx_quantity(ing, meal)] = price_per_gram[ing]

    rows_A, lo, hi, _ = build_constraints(profile_name)

    lower, upper, integrality = build_variable_bounds()

    return milp(
        cost,
        constraints=LinearConstraint(csr_matrix(np.array(rows_A, dtype=float)), lo, hi),
        integrality=integrality,
        bounds=Bounds(lower, upper),
        options={'disp': False, 'time_limit': time_limit}
    )

print('Solver 0 with cost-only defined.')

Solver 0 with cost-only defined.


## Step 10 — Solver 1: Hard Constraint (Three)


**Objective:** Minimize food cost only + hard constraint for repeats.

**Additional hard constraints:**
- **C10:** No same-day repeats — each ingredient appears at most once per day
- **C11:** Max 3 days — each ingredient appears on at most 3 of the 3 days

In [ ]:
def solve_hard_constraint_three(profile_name, time_limit=200):
    """Solver 1: Minimize food cost with hard no-repeat constraints."""

    cost = np.zeros(total_vars)
    for (ing, meal) in eligible_pairs:
        cost[idx_quantity(ing, meal)] = price_per_gram[ing]

    rows_A, lo, hi, _ = build_constraints(profile_name)

    # C10 - No same-day repeats
    for ing in range(num_ingredients):
        for day in range(NUM_DAYS):
            row = np.zeros(total_vars)
            for meal in day_meal_indices[day]:
                if (ing, meal) in pair_index:
                    row[idx_selection(ing, meal)] = 1
            rows_A.append(row); lo.append(0); hi.append(1)

    # C11 - Max 3 days per ingredient
    for ing in range(num_ingredients):
        row = np.zeros(total_vars)
        for day in range(NUM_DAYS):
            row[idx_day_presence(ing, day)] = 1
        rows_A.append(row); lo.append(0); hi.append(3)

    lower, upper, integrality = build_variable_bounds()

    return milp(
        cost,
        constraints=LinearConstraint(csr_matrix(np.array(rows_A, dtype=float)), lo, hi),
        integrality=integrality,
        bounds=Bounds(lower, upper),
        options={'disp': False, 'time_limit': time_limit}
    )

print('Solver 1 with hard constraint with max=3 days defined.')

Solver 1 with hard constraint with max=3 days defined.


## Step 10 — Solver 1: Hard Constraint (Two) - Our Default MILP Model

**Objective:** Minimize food cost only + hard constraint for repeats.

**Additional hard constraints:**
- **C14:** No same-day repeats — each ingredient appears at most once per day
- **C15:** Max 2 days — each ingredient appears on at most 2 of the 3 days

In [ ]:
def solve_hard_constraint(profile_name, time_limit=200):
    """Solver 1: Minimize food cost with hard no-repeat constraints."""

    cost = np.zeros(total_vars)
    for (ing, meal) in eligible_pairs:
        cost[idx_quantity(ing, meal)] = price_per_gram[ing]

    rows_A, lo, hi, _ = build_constraints(profile_name)

    # C10 - No same-day repeats
    for ing in range(num_ingredients):
        for day in range(NUM_DAYS):
            row = np.zeros(total_vars)
            for meal in day_meal_indices[day]:
                if (ing, meal) in pair_index:
                    row[idx_selection(ing, meal)] = 1
            rows_A.append(row); lo.append(0); hi.append(1)

    # C11 - Max 2 days per ingredient
    for ing in range(num_ingredients):
        row = np.zeros(total_vars)
        for day in range(NUM_DAYS):
            row[idx_day_presence(ing, day)] = 1
        rows_A.append(row); lo.append(0); hi.append(2)

    lower, upper, integrality = build_variable_bounds()

    return milp(
        cost,
        constraints=LinearConstraint(csr_matrix(np.array(rows_A, dtype=float)), lo, hi),
        integrality=integrality,
        bounds=Bounds(lower, upper),
        options={'disp': False, 'time_limit': time_limit}
    )

print('Solver 1 with hard constraint with max=2 days defined.')

Solver 1 with hard constraint with max=2 days defined.


## Step 12 — LP Relaxation for Sensitivity Analysis

Relaxes all binary variables to continuous [0, 1]. The dual values (shadow prices) on binding
constraints tell us which constraints are most costly — i.e., which clinical requirements
drive up the meal plan cost the most.

In [ ]:
def solve_lp_relaxation_hard(profile_name, max_days_per_ingredient=2):
    """LP relaxation of Solver 1 for sensitivity analysis."""

    cost = np.zeros(total_vars)
    for (ing, meal) in eligible_pairs:
        cost[idx_quantity(ing, meal)] = price_per_gram[ing]

    rows_A, lo, hi, labels = build_constraints(profile_name)

    # C14 - No same-day repeats
    for ing in range(num_ingredients):
        for day in range(NUM_DAYS):
            row = np.zeros(total_vars)
            for meal in day_meal_indices[day]:
                if (ing, meal) in pair_index:
                    row[idx_selection(ing, meal)] = 1
            rows_A.append(row); lo.append(0); hi.append(1)
            labels.append(f'{ingredient_names[ing]} Day {day+1} max 1/day')

    # C15 - Max days per ingredient
    for ing in range(num_ingredients):
        row = np.zeros(total_vars)
        for day in range(NUM_DAYS):
            row[idx_day_presence(ing, day)] = 1
        rows_A.append(row); lo.append(0); hi.append(max_days_per_ingredient)
        labels.append(f'{ingredient_names[ing]} max {max_days_per_ingredient} days')

    A = np.array(rows_A, dtype=float)
    lower, upper, _ = build_variable_bounds()
    var_bounds = list(zip(lower, upper))

    A_ub_rows = []; b_ub = []; ub_labels = []
    for i in range(len(lo)):
        if hi[i] < 1e8:
            A_ub_rows.append(A[i]); b_ub.append(hi[i])
            ub_labels.append(f'{labels[i]} (upper)')
        if lo[i] > -1e8:
            A_ub_rows.append(-A[i]); b_ub.append(-lo[i])
            ub_labels.append(f'{labels[i]} (lower)')

    result = linprog(cost, A_ub=np.array(A_ub_rows) if A_ub_rows else None,
                     b_ub=b_ub if b_ub else None, bounds=var_bounds, method='highs')
    return result, ub_labels

## Step 13 — Reporting

Helper functions to extract meal plans, nutrient totals, and repeat counts

In [ ]:
def extract_results(x):
    """Extract key metrics from a solution vector."""
    food_cost = sum(price_per_gram[i] * x[idx_quantity(i, j)]
                    for (i, j) in eligible_pairs)

    # Count repeats directly from selection variables
    cross_day = 0
    same_day = 0
    distinct = 0
    for i in range(num_ingredients):
        days_on = [d for d in range(NUM_DAYS)
                   if any(x[idx_selection(i, j)] > 0.5
                          for j in day_meal_indices[d] if (i, j) in pair_index)]
        if days_on:
            distinct += 1
        if len(days_on) > 1:
            cross_day += 1
        for d in range(NUM_DAYS):
            meals_on = sum(1 for j in day_meal_indices[d]
                           if (i, j) in pair_index and x[idx_selection(i, j)] > 0.5)
            if meals_on > 1:
                same_day += 1

    day_totals = []
    for d in range(NUM_DAYS):
        dt = np.zeros(13)
        for j in day_meal_indices[d]:
            for i in range(num_ingredients):
                if (i, j) in pair_index:
                    dt += nutrient_matrix[i] * x[idx_quantity(i, j)]
        day_totals.append(dt)

    meal_cals = [sum(nutrient_matrix[i, 0] * x[idx_quantity(i, j)]
                     for i in range(num_ingredients) if (i, j) in pair_index)
                 for j in range(NUM_MEALS)]

    return {
        'food_cost': food_cost, 'distinct': distinct,
        'cross_day': cross_day, 'same_day': same_day,
        'day_totals': day_totals, 'meal_cals': meal_cals
    }


def print_meal_plan(x, profile_name, label=''):
    """Print a formatted meal plan."""
    print(f'\n{"=" * 70}')
    print(f'  {profile_name} — {label}')
    print(f'{"=" * 70}')

    mcb = meal_cal_bounds.get(profile_name, {})
    total_cost = 0

    for j in range(NUM_MEALS):
        items = []
        meal_cost = 0
        for i in range(num_ingredients):
            if (i, j) in pair_index and x[idx_selection(i, j)] > 0.5:
                grams = x[idx_quantity(i, j)]
                cost = price_per_gram[i] * grams
                meal_cost += cost
                items.append((categories[i], ingredient_names[i], grams, cost))
        total_cost += meal_cost

        items.sort(key=lambda t: ['Protein', 'Grain', 'Vegetable', 'Fruit', 'Fat'].index(t[0])
                if t[0] in ['Protein', 'Grain', 'Vegetable', 'Fruit', 'Fat'] else 5)

        cal = sum(nutrient_matrix[ingredient_names.index(name), 0] * g for _, name, g, _ in items)
        mt = meal_type_codes[j]
        cal_range = f' [{mcb[mt][0]:.0f}-{mcb[mt][1]:.0f}]' if mt in mcb else ''

        print(f'\n  {meal_labels[j]} — ${meal_cost:.2f} ({cal:.0f} kcal{cal_range})')
        for cat, name, grams, cost in items:
            print(f'    [{cat:<9}] {name:<22} {grams:>6.0f}g  ${cost:.2f}')

    print(f'\n  TOTAL: ${total_cost:.2f} / 3 days = ${total_cost/3:.2f} / day')

    bd = profiles[profile_name]
    for d in range(NUM_DAYS):
        dt = np.zeros(13)
        for j in day_meal_indices[d]:
            for i in range(num_ingredients):
                if (i, j) in pair_index:
                    dt += nutrient_matrix[i] * x[idx_quantity(i, j)]
        print(f'\n  Day {d+1} nutrients:')
        for k in range(13):
            lo = bd['min'][k]; hi = bd['max'][k]
            hi_str = 'none' if hi > 1e6 else f'{hi:.0f}'
            mon_flag = ' [monitor]' if k in monitored_nutrients else ''
            status = 'OK' if dt[k] >= lo - 0.5 and (dt[k] <= hi + 0.5 if hi < 1e6 else True) else '!! VIOLATION'
            print(f'    {nutrient_names[k]:<7} {dt[k]:>8.1f} {nutrient_units[k]:<5} [{lo:.0f}-{hi_str}] {status}{mon_flag}')

print('Result extraction defined.')

Result extraction defined.


## Step 14 — MILP All Profiles

Run all solvers across all 5 clinical profiles. Each profile takes between 120 -200 seconds per solver.

## Solver - Cost-Only

In [ ]:
TIME_LIMIT = 180

# Solver 0 - Cost-only
print('=' * 90)
print('  SOLVER 0: Minimize FoodCost')
print('=' * 90)

solution_cost_only = {}
results_cost_only = {}

for profile in profile_names:
    t0 = time.time()
    result = solve_cost_only(profile, time_limit=TIME_LIMIT)
    elapsed = time.time() - t0

    if result.status not in (0, 1, 3) or result.x is None:
        print(f'  {profile}: INFEASIBLE ({elapsed:.1f}s)')
        continue

    tag = 'OPTIMAL' if result.status == 0 else 'FEASIBLE'
    solution_cost_only[profile] = result.x
    row = extract_results(result.x)
    results_cost_only[profile] = row
    print(f'  {profile}: {tag} — Food cost ${row["food_cost"]:.2f}, '
        f'Distinct {row["distinct"]}/41, Cross-day {row["cross_day"]}, '
        f'Same-day {row["same_day"]}, Time {elapsed:.1f}s')

  SOLVER 0: Minimize FoodCost
  Normal Male: OPTIMAL — Food cost $7.05, Distinct 16/41, Cross-day 10, Same-day 10, Time 96.7s
  Normal Female: FEASIBLE — Food cost $7.31, Distinct 17/41, Cross-day 11, Same-day 9, Time 180.0s
  Diabetic: FEASIBLE — Food cost $7.88, Distinct 21/41, Cross-day 13, Same-day 9, Time 180.0s
  High Cholesterol: OPTIMAL — Food cost $5.75, Distinct 16/41, Cross-day 9, Same-day 12, Time 18.9s
  DASH: OPTIMAL — Food cost $6.06, Distinct 17/41, Cross-day 8, Same-day 13, Time 23.8s


## Solver - Hard Repeats ( Max=3 days)

In [ ]:
TIME_LIMIT = 180

# Solver 1 - Hard Constraints
print('=' * 90)
print('  SOLVER 1: Minimize FoodCost + Hard No-Repeats (Max=3 days)')
print('=' * 90)

solutions_hard_three = {}
results_hard_three = {}

for profile in profile_names:
    t0 = time.time()
    result = solve_hard_constraint_three(profile, time_limit=TIME_LIMIT)
    elapsed = time.time() - t0

    if result.status not in (0, 1, 3) or result.x is None:
        print(f'  {profile}: INFEASIBLE ({elapsed:.1f}s)')
        continue

    tag = 'OPTIMAL' if result.status == 0 else 'FEASIBLE'
    solutions_hard_three[profile] = result.x
    row = extract_results(result.x)
    results_hard_three[profile] = row
    print(f'  {profile}: {tag} — Food cost ${row["food_cost"]:.2f}, '
        f'Distinct {row["distinct"]}/41, Cross-day {row["cross_day"]}, '
        f'Same-day {row["same_day"]}, Time {elapsed:.1f}s')

  SOLVER 1: Minimize FoodCost + Hard No-Repeats (Max=3 days)
  Normal Male: FEASIBLE — Food cost $9.93, Distinct 24/41, Cross-day 17, Same-day 0, Time 180.1s
  Normal Female: FEASIBLE — Food cost $10.67, Distinct 26/41, Cross-day 14, Same-day 0, Time 180.0s
  Diabetic: FEASIBLE — Food cost $9.57, Distinct 24/41, Cross-day 16, Same-day 0, Time 180.0s
  High Cholesterol: OPTIMAL — Food cost $8.28, Distinct 23/41, Cross-day 17, Same-day 0, Time 64.1s
  DASH: OPTIMAL — Food cost $8.63, Distinct 23/41, Cross-day 18, Same-day 0, Time 77.8s


## Solver - Hard Repeats (Max=2 days)

In [ ]:
TIME_LIMIT = 180

# Solver 1 - Hard Constraints
print('=' * 90)
print('  SOLVER 1: Minimize FoodCost + Hard No-Repeats')
print('=' * 90)

solutions_hard = {}
results_hard = {}

for profile in profile_names:
    t0 = time.time()
    result = solve_hard_constraint(profile, time_limit=TIME_LIMIT)
    elapsed = time.time() - t0

    if result.status not in (0, 1, 3) or result.x is None:
        print(f'  {profile}: INFEASIBLE ({elapsed:.1f}s)')
        continue

    tag = 'OPTIMAL' if result.status == 0 else 'FEASIBLE'
    solutions_hard[profile] = result.x
    row = extract_results(result.x)
    results_hard[profile] = row
    print(f'  {profile}: {tag} — Food cost ${row["food_cost"]:.2f}, '
        f'Distinct {row["distinct"]}/41, Cross-day {row["cross_day"]}, '
        f'Same-day {row["same_day"]}, Time {elapsed:.1f}s')

  SOLVER 1: Minimize FoodCost + Hard No-Repeats
  Normal Male: FEASIBLE — Food cost $9.81, Distinct 24/41, Cross-day 17, Same-day 0, Time 180.0s
  Normal Female: FEASIBLE — Food cost $10.67, Distinct 26/41, Cross-day 14, Same-day 0, Time 180.0s
  Diabetic: FEASIBLE — Food cost $9.30, Distinct 24/41, Cross-day 17, Same-day 0, Time 180.1s
  High Cholesterol: OPTIMAL — Food cost $8.28, Distinct 23/41, Cross-day 17, Same-day 0, Time 77.3s
  DASH: OPTIMAL — Food cost $8.63, Distinct 23/41, Cross-day 18, Same-day 0, Time 89.2s


## Step 15 — Meal Plans

In [ ]:
print('SOLVER 0 — MEAL PLANS (Cost-Only)')
for profile in solution_cost_only:
    print_meal_plan(solution_cost_only[profile], profile, 'Solver 0 Cost-only')

SOLVER 0 — MEAL PLANS (Cost-Only)

  Normal Male — Solver 0 Cost-only

  Day1-B — $0.58 (700 kcal [700-875])
    [Protein  ] Fortified Milk            200g  $0.12
    [Grain    ] Granola                   100g  $0.25
    [Fruit    ] Banana                     80g  $0.13
    [Fat      ] Flaxseeds                  14g  $0.08

  Day1-L — $1.27 (800 kcal [800-1000])
    [Protein  ] Egg                       100g  $0.33
    [Grain    ] Whole Wheat Bread         120g  $0.22
    [Vegetable] Tomatoes                  147g  $0.12
    [Fruit    ] Banana                    100g  $0.16
    [Fat      ] Almonds                    40g  $0.44

  Day1-D — $0.67 (500 kcal [500-625])
    [Protein  ] Lentils                   100g  $0.15
    [Grain    ] White Rice                172g  $0.12
    [Vegetable] Kale                      181g  $0.27
    [Fruit    ] Banana                     80g  $0.13

  Day2-B — $0.58 (700 kcal [700-875])
    [Protein  ] Fortified Milk            250g  $0.15
    [Grain    ] G

In [ ]:
print('SOLVER 1 — MEAL PLANS (Hard Constraints Max=3 days )')
for profile in solutions_hard_three:
    print_meal_plan(solutions_hard_three[profile], profile, 'Solver 1 (Hard) 3 Days Max')

SOLVER 1 — MEAL PLANS (Hard Constraints Max=3 days )

  Normal Male — Solver 1 (Hard) 3 Days Max

  Day1-B — $0.77 (700 kcal [700-875])
    [Protein  ] Fortified Milk            202g  $0.12
    [Grain    ] Granola                   100g  $0.25
    [Fruit    ] Pineapple                  80g  $0.28
    [Fat      ] Flaxseeds                  19g  $0.12

  Day1-L — $1.26 (800 kcal [800-1000])
    [Protein  ] Egg                       100g  $0.33
    [Grain    ] Whole Wheat Bread         120g  $0.22
    [Vegetable] Tomatoes                  147g  $0.12
    [Fruit    ] Banana                    145g  $0.23
    [Fat      ] Almonds                    33g  $0.37

  Day1-D — $0.70 (500 kcal [500-625])
    [Protein  ] Tofu                      100g  $0.08
    [Grain    ] Spaghetti                 198g  $0.08
    [Vegetable] Carrots                   198g  $0.14
    [Fruit    ] Mango                      80g  $0.40

  Day2-B — $0.89 (705 kcal [700-875])
    [Protein  ] Cottage Cheese            18

In [ ]:
print('SOLVER 1 — MEAL PLANS (Hard Constraints Max=2 days)')
for profile in solutions_hard:
    print_meal_plan(solutions_hard[profile], profile, 'Solver 1 (Hard) 2 Days Max')

SOLVER 1 — MEAL PLANS (Hard Constraints Max=2 days)

  Normal Male — Solver 1 (Hard) 2 Days Max

  Day1-B — $0.77 (700 kcal [700-875])
    [Protein  ] Fortified Milk            227g  $0.14
    [Grain    ] Granola                   100g  $0.25
    [Fruit    ] Pineapple                  80g  $0.28
    [Fat      ] Flaxseeds                  18g  $0.11

  Day1-L — $0.92 (800 kcal [800-1000])
    [Protein  ] Tofu                      100g  $0.08
    [Grain    ] White Rice                125g  $0.09
    [Vegetable] Kale                      150g  $0.23
    [Fruit    ] Banana                    145g  $0.23
    [Fat      ] Peanut Butter              60g  $0.30

  Day1-D — $0.81 (500 kcal [500-625])
    [Protein  ] Lentils                   100g  $0.15
    [Grain    ] Spaghetti                 200g  $0.08
    [Vegetable] Tomatoes                  200g  $0.16
    [Fruit    ] Mango                      83g  $0.42

  Day2-B — $0.89 (700 kcal [700-875])
    [Protein  ] Cottage Cheese            188

## Step 16 — Nutrient Compliance Check

In [ ]:
TOL = 0.5
for label, store, res_store in [('Solver 0', solution_cost_only, results_cost_only),
                                ('Solver 1 - Max 3 days', solutions_hard_three, results_hard_three),
                                ('Solver 1 - Max 2 days', solutions_hard, results_hard)]:
    print(f'\n{"=" * 85}')
    print(f'  Nutrient Compliance — {label} (per day)')
    print(f'{"=" * 85}')
    for profile in store:
        row = res_store[profile]
        bd = profiles[profile]
        print(f'\n  {profile}:')
        for d in range(NUM_DAYS):
            dt = row['day_totals'][d]
            print(f'    Day {d+1}:')
            for k in range(13):
                mon_flag = ' [monitor]' if k in monitored_nutrients else ''
                lo = bd['min'][k]; hi = bd['max'][k]; actual = dt[k]
                hi_str = 'none' if hi > 1e6 else f'{hi:.0f}'
                status = 'OK' if actual >= lo - TOL and (actual <= hi + TOL if hi < 1e6 else True) else '!! VIOLATION'
                print(f'      {nutrient_names[k]:<7} {actual:>8.1f} {nutrient_units[k]:<5} [{lo:.0f}-{hi_str}] {status}{mon_flag}')


  Nutrient Compliance — Solver 0 (per day)

  Normal Male:
    Day 1:
      Cal       2000.0 kcal  [0-none] OK
      Prot        76.2 g     [75-112] OK
      Fat         66.0 g     [66-100] OK
      Carb       283.4 g     [275-412] OK
      Fib         42.6 g     [30-45] OK
      Ca         801.3 mg    [800-1000] OK
      Fe          17.2 mg    [8-45] OK
      Na         946.0 mg    [0-2300] OK
      K         3789.6 mg    [2720-4080] OK
      VitA       735.9 mcg   [0-none] OK [monitor]
      VitD         7.4 mcg   [5-33] OK
      Sugar       80.0 g     [0-none] OK [monitor]
      SatFat      10.0 g     [0-25] OK
    Day 2:
      Cal       2000.0 kcal  [0-none] OK
      Prot        93.7 g     [75-112] OK
      Fat         66.0 g     [66-100] OK
      Carb       275.1 g     [275-412] OK
      Fib         44.8 g     [30-45] OK
      Ca         811.7 mg    [800-1000] OK
      Fe          19.9 mg    [8-45] OK
      Na        1080.1 mg    [0-2300] OK
      K         3071.2 mg    [2720-408

## Step 17 — LP Relaxation & Sensitivity Analysis

The LP relaxation drops all integrality requirements. The **shadow prices** (dual values)
on binding constraints reveal which clinical requirements are most expensive to satisfy
i.e., which constraints, if relaxed slightly, would reduce the meal plan cost the most.

In [ ]:
print('LP Relaxation Results')
print('=' * 85)

for profile in profile_names:
    lp_result, constraint_labels = solve_lp_relaxation_hard(profile)

    if lp_result.success:
        print(f'\n  {profile}: LP cost = ${lp_result.fun:.2f}')

        if hasattr(lp_result, 'ineqlin') and lp_result.ineqlin.marginals is not None:
            duals = lp_result.ineqlin.marginals
            abs_duals = np.abs(duals)
            top_indices = np.argsort(-abs_duals)[:10]
            print(f'    Top 10 binding constraints (largest shadow prices):')
            for rank, idx in enumerate(top_indices):
                if abs_duals[idx] < 1e-6:
                    break
                label = constraint_labels[idx] if idx < len(constraint_labels) else f'row {idx}'
                print(f'      #{rank+1}: shadow price = {duals[idx]:>8.4f}  {label}')
        else:
            print('    (Shadow prices not available)')
    else:
        print(f'\n  {profile}: LP INFEASIBLE')

LP Relaxation Results

  Normal Male: LP cost = $6.83
    Top 10 binding constraints (largest shadow prices):
      #1: shadow price =  -0.3527  Day2-B exactly 1 fruit (lower)
      #2: shadow price =  -0.3527  Day3-B exactly 1 fruit (lower)
      #3: shadow price =  -0.3527  Day1-B exactly 1 fruit (lower)
      #4: shadow price =  -0.3483  Day1-D exactly 1 fruit (lower)
      #5: shadow price =  -0.3483  Day3-D exactly 1 fruit (lower)
      #6: shadow price =  -0.3483  Day2-D exactly 1 fruit (lower)
      #7: shadow price =  -0.3290  Day2-L exactly 1 fruit (lower)
      #8: shadow price =  -0.3290  Day3-L exactly 1 fruit (lower)
      #9: shadow price =  -0.3290  Day1-L exactly 1 fruit (lower)
      #10: shadow price =  -0.3091  Day1-L >=1 protein (lower)

  Normal Female: LP cost = $6.77
    Top 10 binding constraints (largest shadow prices):
      #1: shadow price =  -0.4050  Day1-B exactly 1 fruit (lower)
      #2: shadow price =  -0.4050  Day3-B exactly 1 fruit (lower)
      #3: s

## Step 18 — Solver Comparison

In [ ]:
print('=' * 95)
print('  SOLVER COMPARISON')
print('=' * 95)
print(f"\n  {'Profile':<22} {'Metric':<22} {'Solver 0 (Cost-Only)':>15} {'Solver 1 (Hard 1 Day Max)':>15} {'Solver 1 (Hard 2 Day Max)':>15}")
print('  ' + '─' * 70)

for profile in profile_names:
    r0 = results_cost_only.get(profile)
    r1 = results_hard_three.get(profile)
    r2 = results_hard.get(profile)

    if r0:
        print(f"  {profile:<22} {'Food Cost ($)':<22} {r0['food_cost']:>12.2f} ($)", end='')
        print(f" {r1['food_cost']:>12.2f}($)" , end='' if r1 else '  INFEASIBLE')
        print(f" {r2['food_cost']:>12.2f}($)" if r2 else '  INFEASIBLE')
        print(f"  {'':<22} {'Distinct (/41)':<22} {r0['distinct']:>15}", end='')
        print(f" {r1['distinct']:>15}" , end='' if r1 else '')
        print(f" {r2['distinct']:>15}" if r2 else '')
        print(f"  {'':<22} {'Cross-day Repeats':<22} {r0['cross_day']:>15}", end='')
        print(f" {r1['cross_day']:>15}" , end='' if r1 else '')
        print(f" {r2['cross_day']:>15}" if r2 else '')
        print(f"  {'':<22} {'Same-day Repeats':<22} {r0['same_day']:>15}", end='')
        print(f" {r1['same_day']:>15}" , end='' if r1 else '')
        print(f" {r2['same_day']:>15}" if r2 else '')
        print()

  SOLVER COMPARISON

  Profile                Metric                 Solver 0 (Cost-Only) Solver 1 (Hard 1 Day Max) Solver 1 (Hard 2 Day Max)
  ──────────────────────────────────────────────────────────────────────
  Normal Male            Food Cost ($)                  7.05 ($)         9.93($)         9.81($)
                         Distinct (/41)                      16              24              24
                         Cross-day Repeats                   10              17              17
                         Same-day Repeats                    10               0               0

  Normal Female          Food Cost ($)                  7.31 ($)        10.67($)        10.67($)
                         Distinct (/41)                      17              26              26
                         Cross-day Repeats                   11              14              14
                         Same-day Repeats                     9               0               0

  Diabetic   

## Step 19a — Store Results in DataFrames

All solver results stored in structured DataFrames for easy analysis and custom visualizations.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# --- Build meal plan DataFrames for each solver ---
def build_meal_df(solutions, solver_label):
    """Build a DataFrame with one row per ingredient per meal from a solutions dict."""
    rows = []
    for profile, x in solutions.items():
        for j in range(NUM_MEALS):
            for i in range(num_ingredients):
                if (i, j) in pair_index and x[idx_selection(i, j)] > 0.5:
                    grams = x[idx_quantity(i, j)]
                    cal = nutrient_matrix[i, 0] * grams
                    prot = nutrient_matrix[i, 1] * grams
                    fat = nutrient_matrix[i, 2] * grams
                    carb = nutrient_matrix[i, 3] * grams
                    cost = price_per_gram[i] * grams
                    rows.append({
                        'Solver': solver_label,
                        'Profile': profile,
                        'Day': (j // 3) + 1,
                        'Meal': meal_labels[j],
                        'Meal_Type': meal_type_codes[j],
                        'Ingredient': ingredient_names[i],
                        'Category': categories[i],
                        'Grams': round(grams, 1),
                        'Calories': round(cal, 1),
                        'Protein_g': round(prot, 1),
                        'Fat_g': round(fat, 1),
                        'Carb_g': round(carb, 1),
                        'Cost_CAD': round(cost, 2)
                    })
    return pd.DataFrame(rows)

df_cost_only = build_meal_df(solutions_hard, 'Cost Only')
df_hard_three = build_meal_df(solutions_hard_three, 'Hard Constraint 3 Days Max')
df_hard = build_meal_df(solutions_hard, 'Hard Constraint 2 Days Max')
df_all = pd.concat([df_cost_only, df_hard_three, df_hard], ignore_index=True)

# --- Build daily nutrient summary DataFrames ---
def build_nutrient_df(solutions, results_store, solver_label):
    """Build a DataFrame with daily nutrient totals per profile."""
    rows = []
    for profile in solutions:
        row = results_store[profile]
        bd = profiles[profile]
        for d in range(NUM_DAYS):
            dt = row['day_totals'][d]
            for k in range(13):
                rows.append({
                    'Solver': solver_label,
                    'Profile': profile,
                    'Day': d + 1,
                    'Nutrient': nutrient_names[k],
                    'Unit': nutrient_units[k],
                    'Actual': round(dt[k], 1),
                    'Min': bd['min'][k],
                    'Max': bd['max'][k] if bd['max'][k] < 1e6 else None,
                    'Monitored': k in monitored_nutrients
                })
    return pd.DataFrame(rows)

df_nutrients_cost_only = build_nutrient_df(solution_cost_only, results_cost_only, 'Cost Only')
df_nutrients_hard_three = build_nutrient_df(solutions_hard_three, results_hard_three, 'Hard Constraint 1 Day Max')
df_nutrients_hard = build_nutrient_df(solutions_hard, results_hard, 'Hard Constraint 2 Days Max')
df_nutrients_all = pd.concat([df_nutrients_cost_only, df_nutrients_hard_three, df_nutrients_hard], ignore_index=True)

# Build summary comparison DataFrame
summary_rows = []
for solver_label, res_store in [('Cost Only', results_cost_only), ('Hard Constraint 3 Days Max', results_hard_three), ('Hard Constraint 2 Days Max', results_hard)]:
    for profile, row in res_store.items():
        summary_rows.append({
            'Solver': solver_label,
            'Profile': profile,
            'Food_Cost_3d': round(row['food_cost'], 2),
            'Food_Cost_Daily': round(row['food_cost'] / 3, 2),
            'Distinct_Ingredients': row['distinct'],
            'Cross_Day_Repeats': row['cross_day'],
            'Same_Day_Repeats': row['same_day']
        })
df_summary = pd.DataFrame(summary_rows)

print(f'Meal plan rows: {len(df_all)} ({len(df_cost_only)} cost-only +  {len(df_hard_three)} hard (3 Days Max) + {len(df_hard)} hard (2 Days Max)')
print(f'Nutrient rows: {len(df_nutrients_all)}')
print(f'Summary rows: {len(df_summary)}')
print()
print(df_summary.to_string(index=False))

Meal plan rows: 608 (203 cost-only +  202 hard (3 Days Max) + 203 hard (2 Days Max)
Nutrient rows: 585
Summary rows: 15

                    Solver          Profile  Food_Cost_3d  Food_Cost_Daily  Distinct_Ingredients  Cross_Day_Repeats  Same_Day_Repeats
                 Cost Only      Normal Male          7.05             2.35                    16                 10                10
                 Cost Only    Normal Female          7.31             2.44                    17                 11                 9
                 Cost Only         Diabetic          7.88             2.63                    21                 13                 9
                 Cost Only High Cholesterol          5.75             1.92                    16                  9                12
                 Cost Only             DASH          6.06             2.02                    17                  8                13
Hard Constraint 3 Days Max      Normal Male          9.93             3.31 

## Step 19b — Store Results in Excel

In [ ]:
results_filename = "MILP_Model_Results.xlsx"

with pd.ExcelWriter(results_filename) as writer:
  df_all.to_excel(writer, sheet_name='Meal_Plans', index=False)
  df_nutrients_all.to_excel(writer, sheet_name='Meal_Plan_Nutrients', index=False)
  df_summary.to_excel(writer, sheet_name='Solver_Summary', index=False)
  print('Store Solver results in Excel')


Store Solver results in Excel


## Step 20 — Visualization 1: Cost & Diversity Comparison

Side-by-side comparison of food cost and ingredient diversity across profiles and solvers.

In [ ]:
from plotly.subplots import make_subplots

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=('3-Day Food Cost (CAD)', 'Distinct Ingredients Used'),
                    horizontal_spacing=0.15)

colors = {'Cost Only':' #069494', 'Hard Constraint 3 Days Max': '#FFFF00','Hard Constraint 2 Days Max': '#2196F3'}

for solver in ['Cost Only', 'Hard Constraint 3 Days Max', 'Hard Constraint 2 Days Max']:
    df_s = df_summary[df_summary['Solver'] == solver]
    fig.add_trace(go.Bar(
        name=f'{solver}',
        legendgroup=solver,
        x=df_s['Profile'], y=df_s['Food_Cost_3d'],
        marker_color=colors[solver],
        text=df_s['Food_Cost_3d'].apply(lambda v: f'${v:.2f}'),
        textposition='outside',
        showlegend=True
    ), row=1, col=1)

    fig.add_trace(go.Bar(
        name=f'{solver}',
        legendgroup=solver,
        x=df_s['Profile'], y=df_s['Distinct_Ingredients'],
        marker_color=colors[solver],
        text=df_s['Distinct_Ingredients'].apply(lambda v: f'{v}/41'),
        textposition='outside',
        showlegend=False
    ), row=1, col=2)

fig.update_layout(
    title='Solver Comparison: Cost vs Diversity',
    barmode='group', height=450, width=1000,
    legend=dict(orientation='h', yanchor='bottom', y=1.08, xanchor='center', x=0.5)
)
fig.update_yaxes(title_text='Cost (CAD)', row=1, col=1)
fig.update_yaxes(title_text='Distinct Ingredients', row=1, col=2)
fig.show()

## Step 21 — Visualization 2: Nutrient Compliance Bars

For each profile, shows how daily nutrient actuals compare to their allowed range.
One chart per solver. Only non-monitored nutrients with finite bounds are shown.

In [ ]:
for solver_label, df_nut in [('Cost Only', df_nutrients_cost_only),('Hard Constraint (3 days)', df_nutrients_hard_three),('Hard Constraint', df_nutrients_hard)]:
    for profile in profile_names:
        df_p = df_nut[(df_nut['Profile'] == profile) & (~df_nut['Monitored']) & (df_nut['Max'].notna())]
        if df_p.empty:
            continue

        # Average across 3 days for cleaner display
        df_avg = df_p.groupby('Nutrient').agg(
            Actual=('Actual', 'mean'),
            Min=('Min', 'first'),
            Max=('Max', 'first'),
            Unit=('Unit', 'first')
        ).reset_index()

        # Normalize: show as % of range (0% = at min, 100% = at max)
        df_avg['Range'] = df_avg['Max'] - df_avg['Min']
        df_avg['Pct'] = ((df_avg['Actual'] - df_avg['Min']) / df_avg['Range'] * 100).clip(0, 150)
        df_avg['Label'] = df_avg.apply(
            lambda row: f"{row['Actual']:.0f} {row['Unit']} [{row['Min']:.0f}-{row['Max']:.0f}]", axis=1)

        fig = go.Figure()
        fig.add_trace(go.Bar(
            y=df_avg['Nutrient'], x=df_avg['Pct'],
            orientation='h',
            marker_color=['#4CAF50' if 0 <= p <= 100 else '#F44336' for p in df_avg['Pct']],
            text=df_avg['Label'],
            textposition='outside'
        ))

        # Add the 0-100% feasible zone
        fig.add_vrect(x0=0, x1=100, fillcolor='green', opacity=0.05, line_width=0)
        fig.add_vline(x=0, line_dash='dash', line_color='gray')
        fig.add_vline(x=100, line_dash='dash', line_color='gray')

        fig.update_layout(
            title=f'{profile} — Nutrient Compliance ({solver_label})',
            xaxis_title='Position in Allowed Range (0%=Min, 100%=Max)',
            height=350, width=800,
            margin=dict(l=80, r=150)
        )
        fig.show()

## Step 22 — Visualization 3: Meal Plan Heatmaps

Shows which ingredients appear at each meal across the 3-day plan. Each profile gets a heatmap per solver, with color intensity representing grams.

In [ ]:
for solver_label, df_meals in [('Hard Constraint', df_hard)]:
    for profile in profile_names:
        df_p = df_meals[df_meals['Profile'] == profile]
        if df_p.empty:
            continue

        # Pivot: ingredients as rows, meals as columns, grams as values
        pivot = df_p.pivot_table(index='Ingredient', columns='Meal', values='Grams', fill_value=0)

        # Order columns by meal sequence
        col_order = [m for m in meal_labels if m in pivot.columns]
        pivot = pivot[col_order]

        # Only show ingredients that appear at least once
        pivot = pivot.loc[pivot.sum(axis=1) > 0]

        # Sort by category for readability
        ing_cats = {name: cat for name, cat in zip(ingredient_names, categories)}
        cat_order = ['Protein', 'Grain', 'Vegetable', 'Fruit', 'Fat']
        pivot['_cat'] = pivot.index.map(lambda n: cat_order.index(ing_cats.get(n, 5)))
        pivot = pivot.sort_values('_cat').drop('_cat', axis=1)

        # Add category labels to ingredient names
        pivot.index = [f'[{ing_cats.get(n, "?")}] {n}' for n in pivot.index]

        fig = go.Figure(data=go.Heatmap(
            z=pivot.values,
            x=pivot.columns.tolist(),
            y=pivot.index.tolist(),
            colorscale='YlOrRd',
            text=pivot.values.astype(int).astype(str),
            texttemplate='%{text}g',
            hovertemplate='%{y}<br>%{x}: %{z:.0f}g<extra></extra>',
            colorbar=dict(title='Grams')
        ))

        fig.update_layout(
            title=f'{profile} — Meal Plan ({solver_label})',
            xaxis_title='Meal',
            yaxis=dict(autorange='reversed'),
            height=max(400, len(pivot) * 25 + 100),
            width=900,
            margin=dict(l=200)
        )
        fig.show()